# FujiCV — Synthetic Image Regression on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dsabarinathan/fujicv/blob/main/examples/colab_regression.ipynb)

Trains **ResNet-18** on a **synthetic regression dataset** (mean pixel brightness).
No external downloads needed — images are generated in-notebook.
All results saved to `/content/results_regression/`.

> **Tip:** Set runtime to GPU → Runtime → Change runtime type → **T4 GPU**

## 1. Install FujiCV

In [ ]:
!pip install fujicv timm -q
print('Done.')

## 2. Imports & config

In [ ]:
import logging
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader

import fujicv
from fujicv.engine.trainer import Trainer
from fujicv.eval.plots import plot_loss_curves
from fujicv.losses.regression import MSELoss
from fujicv.metrics.regression import MAE, RMSE, R2Score
from fujicv.models.builder import ModelBuilder
from fujicv.utils.seed import set_seed

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

SEED       = 42
IMAGE_SIZE = 32
BATCH_SIZE = 64
EPOCHS     = 10
LR         = 1e-3
OUTPUT_DIR = '/content/results_regression'

os.makedirs(OUTPUT_DIR, exist_ok=True)
set_seed(SEED)
device = fujicv.get_device()
print(f'Device: {device}')

## 3. Generate synthetic dataset

We create 1000 random 32×32 RGB images. The regression target is the **mean pixel brightness** normalised to [0, 1]. No download needed.

In [ ]:
np.random.seed(SEED)
rows = []
img_dir = '/content/synthetic_regression/images'
os.makedirs(img_dir, exist_ok=True)

for i in range(1000):
    arr = np.random.randint(0, 256, (32, 32, 3), dtype=np.uint8)
    label = float(arr.mean() / 255.0)
    fname = f'img_{i:04d}.jpg'
    Image.fromarray(arr).save(f'{img_dir}/{fname}')
    rows.append({'filename': fname, 'brightness': label})

df = pd.DataFrame(rows)
df.to_csv('/content/synthetic_regression/labels.csv', index=False)
print(f'Generated {len(df)} images. Label range: {df.brightness.min():.3f} – {df.brightness.max():.3f}')
df.head()

## 4. Build DataLoaders via CSVImageDataset

In [ ]:
from fujicv.data.csv_dataset import CSVImageDataset
from fujicv.data.transforms import get_train_transforms, get_val_transforms
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=SEED)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

train_ds = CSVImageDataset(
    csv_path='/content/synthetic_regression/labels.csv',
    img_dir=img_dir,
    label_col='brightness',
    task='regression',
    transform=get_train_transforms(IMAGE_SIZE, level='light'),
    subset_df=train_df,
)
val_ds = CSVImageDataset(
    csv_path='/content/synthetic_regression/labels.csv',
    img_dir=img_dir,
    label_col='brightness',
    task='regression',
    transform=get_val_transforms(IMAGE_SIZE),
    subset_df=val_df,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,}')

## 5. Build model

In [ ]:
model = ModelBuilder(
    backbone_name='resnet18',
    backbone_source='timm',
    pretrained=True,
    task='regression',
    num_outputs=1,
    image_size=IMAGE_SIZE,
).build()

total = sum(p.numel() for p in model.parameters()) / 1e6
print(f'ResNet-18 regression | {total:.1f}M parameters')

## 6. Train

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=MSELoss(),
    metrics={'mae': MAE(), 'rmse': RMSE(), 'r2': R2Score()},
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=EPOCHS,
    task='regression',
    output_dir=OUTPUT_DIR,
    monitor_metric='val_loss',
    mixed_precision=True,
    early_stopping_patience=5,
)

history = trainer.train()

## 7. Training curves

In [ ]:
fig = plot_loss_curves(history)
fig.savefig(f'{OUTPUT_DIR}/loss_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved loss curves.')

## 8. Predicted vs actual scatter plot

In [ ]:
model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for imgs, labels in val_loader:
        preds = model(imgs.to(device)).squeeze(1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_targets.extend(labels.numpy().tolist())

all_preds   = np.array(all_preds)
all_targets = np.array(all_targets)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(all_targets, all_preds, alpha=0.5, s=20)
lims = [min(all_targets.min(), all_preds.min()), max(all_targets.max(), all_preds.max())]
ax.plot(lims, lims, 'r--', label='Perfect prediction')
ax.set_xlabel('Actual brightness')
ax.set_ylabel('Predicted brightness')
ax.set_title('Predicted vs Actual — Synthetic Regression')
ax.legend()
plt.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/scatter_pred_vs_actual.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved scatter plot.')

## 9. Summary

In [ ]:
best_loss = min(history.metrics.get('val_loss', [float('inf')]))
best_mae  = min(history.metrics.get('val_mae',  [float('inf')]))
best_r2   = max(history.metrics.get('val_r2',   [0.0]))

print(f"""
========================================
 FujiCV Synthetic Regression Results
========================================
 Model      : ResNet-18 (pretrained)
 Task       : Brightness regression
 Epochs     : {EPOCHS}  |  Device: {device}
 Best Val MSE  : {best_loss:.5f}
 Best Val MAE  : {best_mae:.5f}
 Best Val R2   : {best_r2:.4f}
========================================
 Saved to /content/results_regression/
   loss_curves.png   scatter_pred_vs_actual.png
   best.pt   history.csv
========================================
""")

for f in sorted(os.listdir(OUTPUT_DIR)):
    kb = os.path.getsize(f'{OUTPUT_DIR}/{f}') / 1024
    print(f'  {f:<40} {kb:6.1f} KB')

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/fujicv_regression_results', 'zip', OUTPUT_DIR)
files.download('/content/fujicv_regression_results.zip')